# 第9周 · 周二：DID 双重差分——利用"趋势差"识别因果\n\n> 难度：⭐⭐⭐⭐ | 定位：理解DID的直觉、数学原理和核心假设，能用Python实现标准DID\n> 前置依赖：周一因果推断基础 | 后续用途：周三DID实现与检验、W10-W12综合案例\n> 面试地位：大厂数据科学面试高频题，仅次于AB实验的因果推断方法\n\n---\n\n## 📘 区块1：核心概念与第一性原理\n\n### 1.1 DID的直觉：为什么不能只看"前后变化"\n\n假设某城市1月1日调高了公交票价，你想评估对客流量的影响。\n\n不能只看"调价前后的客流变化"——因为即使不调价，客流量也可能因为以下原因变化：\n- 季节性：1月本来就是出行淡季\n- 宏观因素：经济下行导致出行减少\n- 外部事件：新地铁线路开通分流\n\n这些因素在没有调价的"平行世界"里也存在。\n\nDID的思路：找一个"没调价的城市"（对照组），用它的变化来"模拟"调价城市如果不调价会怎样变化。\n\n```\n           调价前    调价后    前后差\n调价城市     100      90      -10    ← 包含了调价效应 + 自然趋势\n对照城市     100      95       -5    ← 只有自然趋势（没调价）\n\nDID = (-10) - (-5) = -5\n→ 扣掉自然下降的5单位，调价导致额外减少了5单位\n```\n\n**DID = (实验组后 - 实验组前) - (对照组后 - 对照组前)**\n\n这就是"双重差分"——差分的差分（Difference in Differences）。\n\n### 1.2 DID的数学推导：从潜在结果到估计量\n\nDID的核心识别假设是：如果没有处理，实验组和对照组的时间趋势是相同的（平行趋势假设）。\n\n在潜在结果框架下：\n$$\n\begin{aligned}\nDID &= [E(Y_{post}|T=1) - E(Y_{pre}|T=1)] - [E(Y_{post}|T=0) - E(Y_{pre}|T=0)] \\\\\n    &= [E(Y(1)_{post}|T=1) - E(Y(0)_{pre}|T=1)] - [E(Y(0)_{post}|T=0) - E(Y(0)_{pre}|T=0)] \\\\\n    &= \underbrace{E(Y(1)_{post} - Y(0)_{post}|T=1)}_{ATT} + \\\\\n    &\quad \underbrace{[E(Y(0)_{post} - Y(0)_{pre}|T=1) - E(Y(0)_{post} - Y(0)_{pre}|T=0)]}_{平行趋势假设下 = 0}\n\end{aligned}\n$$\n\n推导的核心：在平行趋势假设下，第二项为0，所以DID = ATT（处理组的平均处理效应）。\n\n### 1.3 平行趋势假设——DID的"阿喀琉斯之踵"\n\n**平行趋势假设**（Parallel Trends Assumption）：如果没有处理，实验组和对照组的结果变量会以相同的趋势变化。\n\n这个假设的关键词是"如果没有处理"——它是关于反事实的，因此**无法被直接检验**。\n\n**为什么无法直接检验**：因为我们永远观测不到实验组"如果没被处理"的处理后结果。我们只能看到实验组被处理后的结果，无法知道如果不被处理它本来会怎样。\n\n**能做的间接检验**：\n1. 检查**处理前**两组的变化趋势是否平行（处理前平行→增强可信度，但不能严格证明处理后也平行）\n2. 做**安慰剂检验**（把处理时间提前到实际处理之前→DID应该不显著）\n3. 加**多个对照组**（如果多个未处理组都显示相似的"反事实趋势"，信心增强）\n\n### 1.4 DID的两个关键时间点 vs 多期DID\n\n**经典2×2 DID**（2组×2期）：\n- 只需要一期"处理前"和一期"处理后"\n- 最简单，但无法检验平行趋势（处理前只有一期，无法判断趋势）\n\n**多期DID**（Multiple Periods）：\n- 处理前有多期，处理后有多期\n- 可以用处理前多期数据检验平行趋势\n- 可以画"事件研究图"（Event Study Plot）——这是论文级别的DID标准操作\n\n### 1.5 面试高频考点\n\n**面试官常问**：\n- "DID的核心假设是什么？如何检验？"\n- "什么时候DID不适用？"\n- "DID和AB实验的区别是什么？"\n- "如果平行趋势不成立怎么办？"\n\n**回答要点**：\n- 核心假设=平行趋势：如果没有处理，两组变化趋势相同\n- 不适用的情况：处理前两组趋势已经分化、存在随时间变化的混淆变量\n- 与AB实验的区别：DID不需要随机化，但需要平行趋势假设；AB实验用随机化消除假设\n- 平行趋势不成立时考虑：三重差分(DDD)、合成控制法、或者加入时间趋势控制变量

## 🧩 区块2：知识点详细拆解\n\n### 2.1 DID的回归表达形式\n\n**为什么重要**：回归形式是DID的标准写法，可以加控制变量、获得标准误和置信区间。\n\n**标准2×2 DID回归方程**：\n```\nY_it = β₀ + β₁·Treated_i + β₂·Post_t + β₃·(Treated_i × Post_t) + ε_it\n```\n\n各系数的含义：\n- β₀：对照组处理前的均值（基准）\n- β₁：处理组与对照组的固有差异（不随时间变化）\n- β₂：共同的时间趋势（两组都有的前后变化）\n- **β₃：DID估计量（交互项系数）——我们关心的核心参数**\n\n**回归形式的优势**：\n1. 可以加控制变量：Y_it = β₀ + β₁·Treated_i + β₂·Post_t + β₃·(Treated_i×Post_t) + γ·X_it + ε_it\n2. 可以直接做统计推断：标准误/t值/p值/置信区间\n3. 可以扩展到多期/多组/连续处理强度\n\n### 2.2 固定效应版本的DID\n\n**为什么重要**：固定效应模型是现代DID的标准做法，比2×2回归更灵活、更稳健。\n\n**双向固定效应模型（TWFE）**：\n```\nY_it = α_i + λ_t + β·D_it + ε_it\n```\n\n- α_i：个体固定效应（吸收Treated_i——所有不随时间变化的个体差异）\n- λ_t：时间固定效应（吸收Post_t——所有个体共同的时间冲击）\n- D_it：处理指示变量（个体i在时间t是否被处理）\n- β：处理效应（在平行趋势和其他假设下）\n\n**TWFE优于2×2回归的地方**：\n1. 个体固定效应控制了所有不随时间变化的混淆变量（可观测的+不可观测的）——这是比PSM强的地方\n2. 时间固定效应控制了所有个体共同的时间冲击\n3. 适用于多期数据和交错处理（Staggered Adoption）\n\n### 2.3 交错处理DID的陷阱（新进展！）\n\n**为什么重要**：2020年以来计量经济学的前沿发现——传统的TWFE在交错处理场景下可能严重有偏！\n\n**问题所在**：当不同个体在不同时间接受处理时（如不同城市在不同时间推出政策），TWFE估计的β实际上是所有可能的2×2 DID比较的加权平均，其中某些权重可能是**负的**（!!!），导致估计量严重偏误。\n\n**解决方案（2021年后）**：\n- **Sun & Abraham (2021)**：用"从未被处理"或"最后被处理"的组作为对照组，分别估计每个相对时间点的效应\n- **Callaway & Sant'Anna (2021)**：分组-时间ATT，允许异质性处理效应\n- **Gardner (2022)**：两阶段DID，先估计组别和时间固定效应，再估计处理效应\n- **Borusyak et al. (2023)**：插补法（Imputation Method）\n\n**实际工作中的建议**：如果你面对的是交错处理场景，不要用传统TWFE——用上述新方法中的一种（Python中可以用`drdid`或`linearmodels`包）。\n\n### 2.4 DID的扩展变体\n\n**为什么重要**：不同场景需要不同的DID变体，知道有哪些"武器"可选很重要。\n\n| 变体 | 适用场景 | 核心思路 |\n|------|---------|----------|\n| **三重差分(DDD)** | 平行趋势不成立时 | 再加一层差分（如再加一个对照组）|\n| **连续DID** | 处理不是0/1而是有强度 | 用连续处理强度替代0/1指示变量 |\n| **合成DID** | 一个处理组+多个对照组 | 用合成控制法构建"最优加权"对照组 |\n| **事件研究DID** | 想看到"动态处理效应" | 用相对时间的一组分项变量替代单一交互项 |\n| **模糊DID** | 处理后"部分人遵从" | 用工具变量处理遵从度不完美 |\n\n### 2.5 DID样本量/检验力——做研究前先算清楚\n\n**为什么重要**：如果你要推动公司做一个DID评估，需要提前说明"多大效应才能检测到"。\n\n**DID的检验力取决于**：\n1. **处理效应大小**（想检测的效应越大，越容易检测到）\n2. **样本量**（处理组和对照组的个体数）\n3. **结果变量的方差**（噪音越小，越容易检测）\n4. **组内相关性**（同一个体不同时期的相关性——通常很高，会降低有效样本量）\n\n**关键经验**：DID的有效样本量不是N×T（个体数×时间点数），而是接近N（个体数）——因为同一个体不同时间点的数据高度相关。很多DID研究的"显著性"其实是高估的（标准误被低估了），需要用**聚类标准误**（Clustered Standard Errors）修正。\n\n### 2.6 DID vs 合成控制法（SCM）\n\n**为什么重要**：当一个处理组面对多个对照组时，DID给每个对照组等权重\n→ 可能不合理（有些对照组更像处理组），合成控制法通过数据驱动确定最优权重。\n\n| 维度 | DID | 合成控制法 |\n|------|-----|----------|\n| 对照组权重 | 等权重（或用专家判断） | 数据驱动的最优权重 |\n| 处理前拟合 | 不一定好 | 要求处理前拟合良好 |\n| 外推风险 | 更小（等权平均更稳健） | 更大（最优权重可能过度拟合） |\n| 适用场景 | 有多个"自然相似"的对照组 | 对照组各有特点，不知道哪个最像 |\n\n### 2.7 DID的常见误用和避坑指南\n\n**为什么重要**：DID是用得最多也误用得最多的因果推断方法之一。\n\n**误用1："有前后+有对照就是DID"**→ 不是。必须满足平行趋势假设！随机选一个对照组直接做DID是危险的。\n\n**误用2："只看交互项的显著性，不看事件研究图"**→ 即使交互项显著，如果处理前系数也偏离0，说明平行趋势不成立，结果不可信。\n\n**误用3："用标准误而不是聚类标准误"**→ DID的数据是面板数据，同一个体不同时期的残差高度相关。用普通标准误会严重低估不确定性，容易"假阳性"。\n\n**误用4："处理组和对照组的样本量差太多"**→ 如果处理组只有1个（如1个城市），对照组有100个——传统DID的推断不可靠（因为没有足够的处理组个体来估计组间方差）。

## 💻 区块3：动手练习\n\n### 步骤0：环境准备

In [ ]:
import numpy as np; import pandas as pd\nimport matplotlib.pyplot as plt\nimport statsmodels.formula.api as smf\nimport warnings; warnings.filterwarnings('ignore')\nnp.random.seed(42)\nplt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']\nplt.rcParams['axes.unicode_minus'] = False\nprint('='*60)\nprint('DID双重差分——动手练习')\nprint('='*60)

### 步骤1：基础练习——2×2 DID（两组×两期）\n\n1.1 模拟公交调价场景，手算DID\n1.2 用回归方法估计DID

In [ ]:
# 步骤1.1：手算2×2 DID\nprint('【步骤1.1】手算DID：公交调价对客流量的影响')\nprint('-'*50)\n\nn_cities = 10\n# 5个调价城市(treated=1)，5个未调价城市(treated=0)\ntreated = np.array([1]*5 + [0]*5)\n# 每个城市的基础客流量不同（城市规模不同）\ncity_base = np.random.normal(100, 10, n_cities)\n\n# 真实的因果参数\ntime_effect = -5       # 自然下降（季节因素，所有城市都有）\ntreat_effect = -8      # 调价导致的额外下降（真正的因果效应！）\n\n# 生成面板数据\ndata = []\nfor i in range(n_cities):\n    for t, tag in [(0, '前'), (1, '后')]:\n        y = city_base[i] + time_effect * t + treat_effect * t * treated[i] + np.random.normal(0, 3)\n        data.append({'city': i, 'time': tag, 'treated': treated[i], 'ridership': y})\ndf = pd.DataFrame(data)\n\n# 计算各组的均值\nbefore_t = df[(df.treated == 1) & (df.time == '前')]['ridership'].mean()\nafter_t  = df[(df.treated == 1) & (df.time == '后')]['ridership'].mean()\nbefore_c = df[(df.treated == 0) & (df.time == '前')]['ridership'].mean()\nafter_c  = df[(df.treated == 0) & (df.time == '后')]['ridership'].mean()\n\ndiff_t = after_t - before_t   # 调价城市的变化\ndiff_c = after_c - before_c   # 对照城市的变化\ndid_manual = diff_t - diff_c  # DID！\n\nprint(f'调价城市: {before_t:.1f} → {after_t:.1f} (变化: {diff_t:+.1f})')\nprint(f'对照城市: {before_c:.1f} → {after_c:.1f} (变化: {diff_c:+.1f})')\nprint(f'DID估计: {did_manual:+.1f} (真实效应={treat_effect})')\nprint(f'→ 调价导致客流量额外变化约{did_manual:.0f}单位（接近真实值{treat_effect}）')\n\n# 可视化\nfig, ax = plt.subplots(figsize=(8, 5))\nax.plot(['调价前', '调价后'], [before_t, after_t], 'o-', color='red', linewidth=2, markersize=10, label=f'调价城市(变化{diff_t:+.1f})')\nax.plot(['调价前', '调价后'], [before_c, after_c], 's--', color='blue', linewidth=2, markersize=10, label=f'对照城市(变化{diff_c:+.1f})')\nax.plot(['调价后', '调价后'], [after_c, after_t], ':k', linewidth=1)\nax.annotate(f'DID={did_manual:+.1f}', xy=('调价后', (after_t+after_c)/2), fontsize=12, color='darkgreen')\nax.set_ylabel('客流量'); ax.legend(); ax.set_title('DID可视化：两条线的垂直差就是DID')\nplt.tight_layout(); plt.show()

In [ ]:
# 步骤1.2：用回归估计DID（含统计推断）\nprint('\\n【步骤1.2】回归法：获得标准误和置信区间')\nprint('-'*50)\n\ndf['post'] = (df['time'] == '后').astype(int)\ndf['treated_post'] = df['treated'] * df['post']  # 交互项\n\n# 用statsmodels做回归\nmodel = smf.ols('ridership ~ treated + post + treated_post', data=df).fit()\nprint(model.summary().tables[1])\nprint()\nprint(f'DID估计量(交互项系数) = {model.params["treated_post"]:.2f}')\nprint(f'95% CI: [{model.conf_int().loc["treated_post", 0]:.2f}, {model.conf_int().loc["treated_post", 1]:.2f}]')\nprint(f'p-value: {model.pvalues["treated_post"]:.4f}')\n\n# 对比：只用处理组前后对比（严重有偏！）\nprint()\nprint('对比：朴素前后对比（只用处理组）:')\nnaive = df[df.treated == 1]['ridership'].mean() - 0  # 这不是正确的做法\nprint(f'  朴素估计 = {diff_t:.1f}（包含了自然下降{sanitize_effect}和调价效应{treat_effect}）')\nprint(f'  → 如果只看前后变化，会误以为调价产生了{diff_t:.0f}的影响')\nprint(f'  → 实际上真实影响只有{treat_effect:.0f}，另外{time_effect:.0f}是季节因素')

### 步骤2：进阶练习——多期DID与事件研究\n\n2.1 生成多期面板数据（处理前4期+处理后4期）\n2.2 画事件研究图检验平行趋势

In [ ]:
# 步骤2.1：多期DID——支持平行趋势检验\nprint('\\n【步骤2.1】多期DID：处理前4期+处理后4期')\nprint('-'*50)\n\nn = 10  # 10个城市\nperiods = list(range(-4, 5))  # -4~-1: 处理前, 0: 处理当期, 1~4: 处理后\ntreated = np.array([1]*5 + [0]*5)\ncity_base = np.random.normal(100, 10, n)\ntreat_effect = 8  # 调价后的持续效应\n\n# 生成数据\ndata = []\nfor i in range(n):\n    for period in periods:\n        time_trend = 2 * period  # 共同线性趋势\n        effect_active = 1 if period >= 0 else 0\n        effect = treat_effect * treated[i] * effect_active\n        y = city_base[i] + time_trend + effect + np.random.normal(0, 3)\n        data.append({'city': i, 'period': period, 'treated': treated[i], 'y': y})\ndf_multi = pd.DataFrame(data)\n\n# 每期计算两组差异（近似事件研究）\nprint('事件研究：每期的处理组-对照组差异')\nprint('时期    | 差异      | 解读')\nprint('-'*45)\ndiffs = []\nfor p in periods:\n    sub = df_multi[df_multi.period == p]\n    diff = sub[sub.treated == 1]['y'].mean() - sub[sub.treated == 0]['y'].mean()\n    diffs.append(diff)\n    marker = ''\n    if p == 0:\n        marker = '← 处理开始'\n    elif p < 0:\n        marker = '(处理前→应接近0)'\n    print(f'  {p:+d}     | {diff:+7.2f}  | {marker}')\n\n# 事件研究图\nfig, ax = plt.subplots(figsize=(10, 5))\nax.plot(periods, diffs, 'o-', linewidth=2, markersize=8, color='darkblue')\nax.axvline(x=-0.5, color='red', linestyle='--', linewidth=1.5, label='处理开始')\nax.axhline(y=0, color='gray', linestyle='-', linewidth=0.8)\nax.fill_between([-4, -0.5], -3, 3, alpha=0.1, color='green', label='处理前区域')\nax.set_xlabel('相对处理时间(期)'); ax.set_ylabel('处理组-对照组差异')\nax.set_title('事件研究图：处理前差异≈0 → 平行趋势成立 ✓')\nax.legend(); plt.tight_layout(); plt.show()\n\nprint()\nprint('判断：处理前(p<0)差异接近0 → 平行趋势成立 ✓')\nprint('      处理后差异增大 → 这是处理效应')

In [ ]:
# 步骤2.2：对比"平行趋势成立"vs"平行趋势不成立"的场景\nprint('\\n【步骤2.2】平行趋势不成立时，DID如何被误导')\nprint('-'*50)\n\n# 场景：处理组本来就处于下降趋势（平行趋势不成立！）\ndata_bad = []\nfor i in range(n):\n    for period in periods:\n        time_trend = 2 * period  # 共同趋势\n        # 处理组有额外的差异化趋势（处理组本来就下降更快）\n        divergent_trend = -3 * period * treated[i]  # 恶化趋势\n        effect = (treat_effect * treated[i]) if period >= 0 else 0\n        y = city_base[i] + time_trend + divergent_trend + effect + np.random.normal(0, 3)\n        data_bad.append({'city': i, 'period': period, 'treated': treated[i], 'y': y})\ndf_bad = pd.DataFrame(data_bad)\n\n# 对比两种场景\nfig, axes = plt.subplots(1, 2, figsize=(14, 5))\n\nfor ax_idx, (df_plot, title, is_good) in enumerate([\n    (df_multi, '平行趋势成立 ✓', True),\n    (df_bad, '平行趋势不成立 ✗', False)\n]):\n    ax = axes[ax_idx]\n    for t_val, label, ls in [(1, '处理组', '-'), (0, '对照组', '--')]:\n        sub = df_plot[df_plot.treated == t_val].groupby('period')['y'].mean()\n        ax.plot(sub.index, sub.values, ls, linewidth=2, marker='o', markersize=5, label=label)\n    ax.axvline(x=-0.5, color='red', linestyle=':', linewidth=1.5)\n    ax.set_xlabel('相对处理时间'); ax.set_ylabel('结果变量')\n    ax.set_title(title); ax.legend()\nplt.tight_layout(); plt.show()\n\nprint('左图：处理前两组趋势平行 → DID可信')\nprint('右图：处理前两组趋势就已经分化 → DID会错误地把"本来就存在的分化"归因为处理效应')

### 步骤3：挑战练习——完整DID分析流程\n\n3.1 模拟真实的业务场景（新功能上线）\n3.2 从数据处理到结果解读的完整闭环

In [ ]:
# 步骤3：完整DID分析——某APP上线"直播功能"对用户时长的影响\nprint('\\n【步骤3】完整DID流程：新功能效果评估')\nprint('='*60)\n\nnp.random.seed(2024)\nn_users = 200  # 100个处理组(安卓用户先上线)+100个对照组(iOS用户后上线)\nn_periods = 14  # 7天处理前+7天处理后\n\n# 用户特征\ntreated = np.array([1]*100 + [0]*100)  # 安卓=处理组，iOS=对照组\nuser_base = np.random.normal(45, 10, n_users)  # 每人基础时长不同\n\n# 生成面板数据\ndata_full = []\nfor i in range(n_users):\n    for t in range(n_periods):\n        day = t - 7  # -7~-1: 处理前, 0~6: 处理后\n        post = 1 if day >= 0 else 0\n        # 周末效应\n        weekend_effect = 5 if day % 7 in [5, 6] else 0\n        # 直播功能效应（处理后）\n        live_effect = 8 if (post == 1 and treated[i] == 1) else 0\n        # 自然时间趋势（整体在增长）\n        trend = 0.3 * t\n        y = user_base[i] + trend + weekend_effect + live_effect + np.random.normal(0, 5)\n        data_full.append({'user': i, 'day': day, 'post': post, 'treated': treated[i], 'minutes': y})\ndf_full = pd.DataFrame(data_full)\n\n# 聚合到日级别\ndaily = df_full.groupby(['day', 'treated'])['minutes'].mean().reset_index()\n\n# 分别画处理组和对照组的日趋势\nfig, ax = plt.subplots(figsize=(12, 5))\nfor t_val, label, color, ls in [(1, '安卓(处理组)', 'red', '-'), (0, 'iOS(对照组)', 'blue', '--')]:\n    sub = daily[daily.treated == t_val]\n    ax.plot(sub['day'], sub['minutes'], ls, color=color, linewidth=2, marker='o', markersize=4, label=label)\nax.axvline(x=-0.5, color='green', linestyle=':', linewidth=2, label='直播功能上线')\nax.set_xlabel('相对上线天数'); ax.set_ylabel('日均使用时长(分钟)')\nax.set_title('新功能上线的DID分析：安卓vs iOS')\nax.legend(); plt.tight_layout(); plt.show()\n\n# 做DID回归\ndf_full['treated_post'] = df_full['treated'] * df_full['post']\nmodel_full = smf.ols('minutes ~ treated + post + treated_post', data=df_full).fit()\ndid_coef = model_full.params['treated_post']\ndid_ci = model_full.conf_int().loc['treated_post']\ndid_p = model_full.pvalues['treated_post']\n\nprint(f'\\n完整DID分析结果:')\nprint(f'  DID估计(直播功能对时长的因果效应): {did_coef:.2f}分钟/天')\nprint(f'  95%置信区间: [{did_ci[0]:.2f}, {did_ci[1]:.2f}]')\nprint(f'  p值: {did_p:.4f}')\nprint(f'  真实效应=8分钟/天')\nprint(f'  结论: 直播功能显著提升了用户时长（提升了约{did_coef:.0f}分钟/天）')

## 📊 区块4：业务场景应用\n\n### 场景1：评估某城市"公交免费换乘"政策的效果\n\n**背景**：A城市2024年3月推出"公交-地铁2小时内免费换乘"政策，目标是鼓励公共交通。你有A城市和周边B、C、D三个未推出此政策的城市的月度公交刷卡数据（2023.1-2024.12）。\n\n**分析思路**：\n\n1. **检查平行趋势**：画出2023.1-2024.2（处理前）A城市和B/C/D城市的月客运量趋势图。如果趋势基本平行→DID可以用。\n\n2. **基准DID**：用2×2 DID（2024.2=处理前，2024.3=处理后第一个月）。估计政策的即时效应。\n\n3. **多期DID**：用全部24个月的数据做事件研究，看政策效果是否持续、是否有延迟效应。\n\n4. **合成控制法作为鲁棒性检验**：用B/C/D三个城市加权合成一个"虚拟A城市"，比较真实A和虚拟A在2024.3后的差异。\n\n5. **预期发现**：政策推出后第一个月，A的客运量比合成控制预测值高约8-12%。这个效应在初期最高，之后稳定在5-8%。\n\n**关键决策**：公交公司想知道这个政策"值不值"——带来的额外票务收入是否覆盖了免费换乘的成本。DID估计出的增量乘客×平均票价=增量收入，和政策成本比较。\n\n### 场景2：大促活动的"增量GMV"评估\n\n**背景**：电商平台计划在3月做一次全品类大促。老板想知道大促到底带来了多少"增量GMV"（如果没有大促，这些GMV会不会自然发生？）。\n\n**DID设计思路**：\n\n1. **处理组**：参与大促的品类（或城市/渠道）\n2. **对照组**：没参与大促的品类——但要注意：为什么有些品类没参与？如果是"因为卖得不好所以没参与"，那就是选择偏差，不能直接用DID。\n\n3. **更合理的设计**：选一个"不参与大促的对照平台"（如果公司有多个平台/渠道），或者"在历史上相同时期（去年3月）做对比"。\n\n4. **DID的"时间"维度**：处理前=大促前一周，处理后=大促周。还要关注"大促后一周"——看是否有"透支效应"（大促结束后GMV暴跌）。\n\n5. **结论示例**：大促周GMV提升了30%，但DID估计增量只有18%——因为每年3月本来就是换季消费旺季（自然增长12%）。更关键的发现：大促后一周GMV比正常低8%，说明存在透支效应。\n\n**决策**：如果18%的增量GMV带来的利润 > 大促的营销和折扣成本，大促是划算的。但考虑到透支效应（相当于"借了未来的销售"），可能需要拉长评估周期。\n\n### 场景3：无法用DID的典型案例——教你识别"DID不适用"的信号\n\n**案例**：某公司想评估"涨薪10%"对员工离职率的影响，拿"2023年涨薪的部门"作为处理组，"没涨薪的部门"作为对照组。\n\n**为什么DID不适用**：\n\n1. **反向因果/选择偏差**：公司选择给哪些部门涨薪，本身可能和离职率有关——比如"离职率本来就高的部门"更可能被涨薪（为了留人）。处理前两组趋势很可能不平行。\n\n2. **溢出效应（SUTVA违反）**：没涨薪的部门员工知道了别的部门涨薪，可能产生不满情绪从而增加离职——对照组被"污染"了。\n\n3. **策略建议**：这种场景下，如果可能的话，做一个"随机化的试点涨薪"（AB实验），或者用PSM匹配"本来被涨薪概率相似但因预算原因没被涨薪的部门"。\n\n### 场景4：面试模拟——"用户补贴的效率评估"\n\n**面试官**："我们给一批用户发了优惠券，想评估券的增量效果。你会怎么做？"\n\n**你的回答框架**：\n\n1. "首先问清楚：券是怎么发的？是随机的还是定向的？如果是随机的——那太好了，这是AB实验，直接比较两组即可。如果是定向的——需要因果推断方法。"\n\n2. "如果是定向发放（比如只给"沉睡用户"发），我需要找一个对照组来构建DID。比如：找一批'同样是沉睡用户但因预算限制没被发券的人'，对比发券前后两组消费的变化。"\n\n3. "关键检查：处理前两组趋势是否平行？如果没发券的沉睡用户也在自然回流（因为他们本来就是'曾经活跃过'的用户），那平行趋势可能不成立，DID会高估券的效果。"\n\n4. "我会建议：先做一个小规模随机化实验（随机从沉睡用户中抽取一部分发券），用实验来'校准'观测方法的估计。长期来看，如果实验和DID结果接近→DID可靠；如果差距大→说明有未观测混淆，需要换方法。

## ✅ 区块5：分级检验清单\n\n### 🟢 初级检验（必须能独立完成的）\n- [ ] 能写出DID的计算公式：DID = (后_处理 - 前_处理) - (后_对照 - 前_对照)\n- [ ] 能写出DID的回归方程并解释每个系数的含义\n- [ ] 能用自己的话解释平行趋势假设\n- [ ] 能做2×2 DID的手算和回归\n\n**答案提示**：\n- DID = diff_t - diff_c = (after_t - before_t) - (after_c - before_c)\n- Y = β₀ + β₁·T + β₂·P + β₃·(T×P)；β₃是DID估计量\n- 平行趋势：如果没有处理，两组变化趋势相同\n- 手算=算四个均值做两次差分；回归=交互项的系数\n\n### 🟡 中级检验（能解释给同事听的原理）\n- [ ] 能解释为什么DID不要求所有混淆变量可观测（而PSM要求）\n- [ ] 能画出事件研究图并判断平行趋势是否成立\n- [ ] 能说明DID中"聚类标准误"的必要性\n- [ ] 能解释交错处理DID的偏误来源\n\n**答案提示**：\n- DID通过差分消除了不随时间变化的未观测混淆（个体固定效应）；PSM需要所有混淆被观测并纳入模型\n- 处理前系数应接近0（不显著），处理后系数显著偏离0→平行趋势成立+处理有效应\n- 面板数据中同一体多期的残差相关→普通标准误低估→聚类标准误到个体级别\n- 交错处理下TWFE是"所有2×2比较的加权平均"，其中"已处理组作为对照组"的权重可能为负\n\n### 🔴 高级检验（能举一反三的迁移能力）\n- [ ] 给定一个业务场景，能判断DID是否适用、需要什么数据、如何设计对照组\n- [ ] 能说出DID的三个关键威胁（平行趋势违反/SUTVA违反/组间构成变化）及应对方案\n- [ ] 能区分"政策评估"和"因果推断"的DID在分析要求上的差异\n- [ ] 能用合成控制法的思路增强DID的可信度\n\n**答案提示**：\n- DID适用条件：有前后+有对照+处理前趋势平行；对照组应该和处理组"除了处理外其他方面相似"\n- 三个威胁：(1)平行趋势不成立→事件研究图检验,加控制变量,或换DDD/SCM (2)SUTVA违反→确保对照组不受处理影响 (3)组间构成变化→检查两组在处理前后是否有大规模人员变动\n- 政策评估更关注"总体效应"（对全社会的影响），DID需要关注溢出效应和一般均衡效应\n- 合成控制法用数据驱动权重（而不是等权重），处理前拟合更好→DID的可信度更高

## 🔗 区块6：跨模块连接\n\n| 连接方向 | 关联内容 | 为什么有关 |\n|---------|---------|----------|\n| **← 前置依赖** | 周一因果推断基础（反事实/混淆/选择偏差） | DID的反事实是"对照组时间趋势外推"，理解选择偏差才能理解为什么需要对照组 |\n| **← 前置依赖** | 回归分析（OLS/标准误/假设检验） | DID的标准实现是回归+交互项，统计推断依赖回归框架 |\n| **→ 后续用途** | 周三DID实现与检验（平行趋势检验/安慰剂检验） | 周三深入检验DID假设是否成立 |\n| **→ 后续用途** | W6同期群分析中的留存趋势对比 | 同期群之间的趋势对比本质上是DID的思想 |\n| **业务连接** | 策略评估、大促效果评估、政策评估 | 所有"某事件前后的效果评估"都可以考虑DID |\n\n### 横向连接：与其他分析方法的关联\n\n**与AB实验的连接**：\n- DID可以被视为"带有时间维度的AB实验"——如果随机化成功，DID交互项应该不显著（因为随机化已经保证了基线可比）\n- 实际中常在AB实验中加入DID分析（ANCOVA的一种形式）来提升精度\n\n**与PSM的连接**：\n- PSM+DID：先用PSM匹配处理组和对照组，再做DID——结合两者的优势（PSM控制可观测混淆+DID消除固定效应）\n- 这是实证研究中常见的组合拳\n\n**与机器学习的连接**：\n- 合成控制法用优化方法确定对照组权重→这是DID的ML增强版\n- 矩阵补全法（Matrix Completion）将面板数据视为矩阵，用低秩分解估计反事实→非常前沿

## 📝 区块7：当日小结\n\n> **DID = 双重差分 = (实验组后-前) - (对照组后-前)。它的核心洞察是：对照组的变化包含了"所有共同的时间趋势"，从实验组的变化中扣掉这个趋势，剩下的就是处理的净效应。DID比朴素前后对比强在"剥离了时间趋势"，比朴素组间对比强在"剥离了固有差异"。但DID的有效性依赖于平行趋势假设——处理前两组趋势一致。这就是DID的"阿喀琉斯之踵"：一旦平行趋势不成立，DID就是瞎猜。检验方法：画事件研究图，看处理前各期系数是否接近0。此外，固定效应版本(TWFE)优于朴素2×2回归，但要注意交错处理下的"负权重"问题。聚类标准误是必须的——否则你的p值会过于乐观。**\n\n**三条金句带回家**：\n1. DID = 用对照组的趋势来"模拟"实验组的反事实——前提是两组处理前趋势平行\n2. 事件研究图是DID的"安检"——处理前系数不为0→平行趋势不成立→换方法\n3. TWFE控制了所有不随时间变化的混淆（可观测+不可观测）——这是DID比PSM强大的地方\n\n### 🔖 课后拓展\n\n1. **练习**：模拟一个"平行趋势不成立"的DID场景，计算DID估计和真实效应的差距\n\n2. **思考**：如果处理组有100个城市，对照组只有5个，聚类标准误还可靠吗？(提示：聚类数量少时，聚类标准误本身有偏→考虑wild bootstrap或随机化推断)\n\n3. **预习**：明天要学的DID实现与检验——如何用代码做平行趋势检验和安慰剂检验\n\n4. **面试准备**：\n   - "DID的核心假设是什么？如何检验？"\n   - "为什么DID的回归要用聚类标准误？"\n   - "DID和AB实验的区别是什么？"\n\n---\n\n*03analyst_thinking · Week 9 Day 2 · 2026-05-21*